# Sentiment Analysis Homework

Complete ML pipeline with explanations and error analysis.

In [1]:
import nltk
nltk.download('movie_reviews')

from nltk.corpus import movie_reviews
import random


[nltk_data] Downloading package movie_reviews to
[nltk_data]     C:\Users\levuri\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\movie_reviews.zip.


# Exercise 1: Conceptual Answers

## 1. Model Failure
Lexicon-based models rely on individual word polarity. The phrase *"anything but boring"* contains the word "boring" (negative), but the structure negates it. The model fails because it cannot understand context or negation.

An ML model can learn this by seeing many examples where similar patterns appear and associating them with positive sentiment.

## 2. Data Requirements
I would choose an ML-based approach because jargon is domain-specific.

Main challenge: collecting enough labeled data so the model understands technical vocabulary.

## 3. Multi-class Sentiment
To extend to multi-class:
- Change labels to multiple classes (Happy, Sad, Angry, Surprised)
- Use a classifier that supports multi-class (LogisticRegression does)
- No major pipeline changes needed, just different labels


In [5]:
# Load data
documents = [(list(movie_reviews.words(fileid)), category)
             for category in movie_reviews.categories()
             for fileid in movie_reviews.fileids(category)]

random.shuffle(documents)

# Prepare data
X_text = [" ".join(words) for words, label in documents]
y = [label for words, label in documents]

# Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_text, y, test_size=0.2, random_state=42)

# Pipeline
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words='english')),
    ("clf", LogisticRegression(max_iter=1000))
])

# Train
pipeline.fit(X_train, y_train)

# Evaluate
from sklearn.metrics import accuracy_score

y_pred = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)


Accuracy: 0.8475


In [6]:
# Error Analysis

misclassified = []

for text, true, pred in zip(X_test, y_test, y_pred):
    if true != pred:
        misclassified.append((text, true, pred))

# Show first 2 mistakes
for i in range(2):
    print("Review:", misclassified[i][0][:500])
    print("True:", misclassified[i][1])
    print("Predicted:", misclassified[i][2])
    print("-" * 80)


Review: this film is based on the campy tv show from the 1960 ' s under the same appellation . mind you , most people ( including yours truly ) who will see this movie will not have seen any episodes from the original series . the movie is really a stand alone in that regard . the family robinson . . . lost in space . plot : set in the year 2058 , the family robinson is chosen to sail out into space in search of other planets that might contain the natural resources earth needs in order to survive its f
True: pos
Predicted: neg
--------------------------------------------------------------------------------
Review: the second jackal - based film to come out in 1997 ( the other starring bruce willis was simply entitled the jackal ) , this one stars aidan quinn and donald sutherland , and is directed by a man who hailed from joblo ' s own alma matter , concordia university in montreal , canada . the story is based on the exploits of the real terrorist known as the jackal , but does not p

## Analysis

Possible reasons for misclassification:
- Sarcasm or irony
- Complex negation
- Rare or domain-specific words
- Mixed sentiment in one review
